# 🎥 Notebook 6 — Real-Time Inference & Benchmarking
**Pill Counter | Computer Vision Pipeline**

> **Goal:** Benchmark inference speed on static images and video, visualise frame-level detections, and profile latency at different model sizes and input resolutions.

---
**Sections:**
1. Imports & model load
2. Single-image inference
3. Batch inference benchmark
4. Latency profiling (resolution × model variant)
5. FPS benchmark on video file
6. Camera inference (interactive)
7. Export to ONNX


## 6.1 — Imports & Model Load

In [ ]:
import torch, yaml, time, json, cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

plt.rcParams["figure.dpi"] = 120

with open("config.yaml") as f:
    CFG = yaml.safe_load(f)

MODEL_PATH = Path(CFG["paths"]["trained_models"]) / "best.pt"
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
CONF       = CFG["inference"]["confidence_threshold"]
IOU        = CFG["inference"]["nms_threshold"]

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH} — run Notebook 4 first."
model = YOLO(str(MODEL_PATH))
model.to(DEVICE)
print(f"✅  Model loaded | Device: {DEVICE}")


## 6.2 — Single Image Inference

In [ ]:
with open(Path(CFG["paths"]["processed_data"]) / "dataset.yaml") as f:
    ds = yaml.safe_load(f)

test_dir   = Path(ds["path"]) / ds["test"]
test_imgs  = list(test_dir.glob("*.*"))

assert test_imgs, f"No test images found in {test_dir}"
img_path   = test_imgs[0]
img        = cv2.imread(str(img_path))

# Inference
t0 = time.perf_counter()
result = model(img, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
latency_ms = (time.perf_counter() - t0) * 1000

# Annotate
drawn = img.copy()
for i, box in enumerate(result.boxes):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    conf_v = float(box.conf[0])
    cv2.rectangle(drawn, (x1, y1), (x2, y2), (0, 200, 80), 2)
    cv2.putText(drawn, f"#{i+1} {conf_v:.2f}", (x1, max(y1-5,10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 80), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img,   cv2.COLOR_BGR2RGB)); axes[0].set_title("Input");     axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"Detected {len(result.boxes)} pill(s)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

print(f"Detections : {len(result.boxes)}")
print(f"Latency    : {latency_ms:.1f} ms  ({1000/latency_ms:.1f} FPS theoretical)")


## 6.3 — Batch Inference Benchmark

> Measure throughput on N images (warm-up + timed run).

In [ ]:
BATCH_IMGS = test_imgs[:50]  # ← adjust as needed

# Warm-up (first inference is always slow)
_ = model(cv2.imread(str(BATCH_IMGS[0])), verbose=False)

latencies = []
for p in BATCH_IMGS:
    img = cv2.imread(str(p))
    if img is None: continue
    t0 = time.perf_counter()
    model(img, conf=CONF, iou=IOU, device=DEVICE, verbose=False)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f"Batch size  : {len(latencies)} images")
print(f"Mean latency: {latencies.mean():.1f} ms")
print(f"P50 latency : {np.percentile(latencies, 50):.1f} ms")
print(f"P95 latency : {np.percentile(latencies, 95):.1f} ms")
print(f"P99 latency : {np.percentile(latencies, 99):.1f} ms")
print(f"Throughput  : {1000/latencies.mean():.1f} FPS")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(latencies, color="#4C8BF5", linewidth=1)
ax.axhline(latencies.mean(), color="red", linestyle="--", label=f"Mean {latencies.mean():.1f}ms")
ax.set_xlabel("Image index"); ax.set_ylabel("Latency (ms)")
ax.set_title("Per-Image Inference Latency"); ax.legend()
plt.tight_layout()
plt.savefig("results/visualizations/latency_profile.png", bbox_inches="tight")
plt.show()


## 6.4 — Resolution × Latency Sweep

> Find the sweet spot between accuracy and speed by varying input resolution.

In [ ]:
resolutions  = [320, 416, 512, 640, 768]
mean_latency = []
img = cv2.imread(str(BATCH_IMGS[0]))

for res in resolutions:
    times = []
    for _ in range(20):  # 20 runs per resolution
        t0 = time.perf_counter()
        model(img, imgsz=res, conf=CONF, iou=IOU, device=DEVICE, verbose=False)
        times.append((time.perf_counter() - t0) * 1000)
    mean_latency.append(np.mean(times))
    print(f"  imgsz={res:4d}: {mean_latency[-1]:.1f} ms  ({1000/mean_latency[-1]:.1f} FPS)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(resolutions, mean_latency, "o-", color="#4C8BF5", linewidth=2)
ax.axhline(1000/30, color="orange", linestyle="--", label="30 FPS target (33ms)")
ax.set_xlabel("Input resolution (px)"); ax.set_ylabel("Mean latency (ms)")
ax.set_title("Inference Latency vs Input Resolution"); ax.legend()
plt.tight_layout()
plt.savefig("results/visualizations/resolution_latency.png", bbox_inches="tight")
plt.show()


## 6.5 — Video File Inference

In [ ]:
# ── Replace VIDEO_PATH with your video file ──
VIDEO_PATH  = "data/sample_video.mp4"   # ← update this
OUTPUT_PATH = "results/output_video.mp4"

if not Path(VIDEO_PATH).exists():
    print(f"⚠️  Video not found: {VIDEO_PATH}")
    print("   Skipping video inference. Update VIDEO_PATH to a real file.")
else:
    cap    = cv2.VideoCapture(VIDEO_PATH)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out    = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    frame_latencies = []

    while True:
        ret, frame = cap.read()
        if not ret: break

        t0     = time.perf_counter()
        result = model(frame, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
        lat    = (time.perf_counter() - t0) * 1000
        frame_latencies.append(lat)

        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 80), 2)

        count = len(result.boxes)
        cv2.putText(frame, f"Pills: {count}  {1000/lat:.0f}FPS",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 200, 80), 2)
        out.write(frame)

    cap.release(); out.release()
    print(f"✅  Output saved: {OUTPUT_PATH}")
    print(f"   Frames: {len(frame_latencies)} / {total}")
    print(f"   Mean FPS: {1000/np.mean(frame_latencies):.1f}")


## 6.6 — Camera Inference (Interactive)

> Run this cell in a local Jupyter session with a webcam. Press `q` to quit.

In [ ]:
# ── Requires a display (not suitable for headless servers) ──
RUN_CAMERA = False  # ← set True when running locally with a webcam

if RUN_CAMERA:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌  Cannot open camera 0")
    else:
        print("Camera opened. Press Q to quit.")
        frame_count = 0
        fps_tracker = []

        while True:
            ret, frame = cap.read()
            if not ret: break

            t0     = time.perf_counter()
            result = model(frame, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
            lat    = (time.perf_counter() - t0) * 1000
            fps_tracker.append(lat)

            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cf = float(box.conf[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 80), 2)
                cv2.putText(frame, f"pill {cf:.2f}", (x1, max(y1-5,10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 80), 1)

            pill_count = len(result.boxes)
            cv2.putText(frame, f"Count: {pill_count}  |  {1000/lat:.0f} FPS",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 80), 2)

            cv2.imshow("Pill Counter", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        cap.release()
        cv2.destroyAllWindows()
        print(f"Mean FPS: {1000/np.mean(fps_tracker):.1f}")
else:
    print("RUN_CAMERA=False — set to True and re-run with a webcam connected.")


## 6.7 — Export to ONNX

In [ ]:
EXPORT_FORMAT = "onnx"  # also: "tflite", "engine" (TensorRT), "coreml"
EXPORT_DIR    = Path("models/exported")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

exported = model.export(format=EXPORT_FORMAT, imgsz=CFG["dataset"]["img_size"])
print(f"✅  Model exported: {exported}")
print(f"   Size: {Path(exported).stat().st_size / 1e6:.1f} MB")


## ✅  Notebook 6 Complete

All six notebooks form a complete, reproducible ML pipeline:

| # | Notebook | Status |
|---|----------|--------|
| 1 | Environment Setup | ✅ |
| 2 | Data Loading & EDA | ✅ |
| 3 | Preprocessing & Augmentation | ✅ |
| 4 | Model Training | ✅ |
| 5 | Evaluation & Metrics | ✅ |
| 6 | Real-Time Inference | ✅ |
